In [7]:
import torch
import torch.nn as nn
from PIL import Image
from torchvision import transforms
import timm

# Config
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
NUM_CLASSES = 6
IMAGE_SIZE = 224

CLASS_NAMES = [
    "demodicosis",
    "dermatitis",
    "fungal_infections",
    "healthy",
    "hypersensitivity",
    "ringworm"
]

# Transforms
val_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5]*3, std=[0.5]*3)
])

# Create model with correct architecture
model = timm.create_model(
    "vit_base_patch16_224",
    pretrained=False,
    num_classes=NUM_CLASSES
)
model.to(DEVICE)

def predict_image(image_path, model):
    model.eval()
    image = Image.open(image_path).convert("RGB")
    image = val_transform(image).unsqueeze(0).to(DEVICE)

    with torch.no_grad():
        outputs = model(image)
        pred = outputs.argmax(dim=1).item()

    return CLASS_NAMES[pred]

# Load model
model.load_state_dict(torch.load("best_vit_model.pth", map_location=DEVICE))

# Predict
result = predict_image("demodicosis.jpg", model)
print("Prediction:", result)

Prediction: hypersensitivity
